# Phase 3 — Feature engineering

# Setting up libraries

In [2]:
import pandas as pd
import numpy as np

import datetime as dt
from datetime import date

from dateutil.relativedelta import relativedelta

In [14]:
print("table: online retail transactions")
retail_data = pd.read_csv('../data/interim/cleaned_retail_transactions.csv')
display(retail_data.head())

table: online retail transactions


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,is_cancellation,is_non_product,is_missing_customer,is_customer_cancellation,is_stock_adjustment
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,False,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,False,False,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,False,False,False,False


In [15]:
retail_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1028761 entries, 0 to 1028760
Data columns (total 13 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   invoice                   1028761 non-null  str    
 1   stockcode                 1028761 non-null  str    
 2   description               1028761 non-null  str    
 3   quantity                  1028761 non-null  int64  
 4   invoicedate               1028761 non-null  str    
 5   price                     1028761 non-null  float64
 6   customer_id               797885 non-null   float64
 7   country                   1028761 non-null  str    
 8   is_cancellation           1028761 non-null  bool   
 9   is_non_product            1028761 non-null  bool   
 10  is_missing_customer       1028761 non-null  bool   
 11  is_customer_cancellation  1028761 non-null  bool   
 12  is_stock_adjustment       1028761 non-null  bool   
dtypes: bool(5), float64(2), int64(1), str(

In [16]:
retail_data["customer_id"] = retail_data["customer_id"].astype("Int64")

In [17]:
retail_data["invoicedate"] = pd.to_datetime(retail_data["invoicedate"], errors='coerce')

In [18]:
retail_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1028761 entries, 0 to 1028760
Data columns (total 13 columns):
 #   Column                    Non-Null Count    Dtype         
---  ------                    --------------    -----         
 0   invoice                   1028761 non-null  str           
 1   stockcode                 1028761 non-null  str           
 2   description               1028761 non-null  str           
 3   quantity                  1028761 non-null  int64         
 4   invoicedate               1028761 non-null  datetime64[us]
 5   price                     1028761 non-null  float64       
 6   customer_id               797885 non-null   Int64         
 7   country                   1028761 non-null  str           
 8   is_cancellation           1028761 non-null  bool          
 9   is_non_product            1028761 non-null  bool          
 10  is_missing_customer       1028761 non-null  bool          
 11  is_customer_cancellation  1028761 non-null  bool          
 1

In [19]:
# Revenue calculation
retail_data["revenue"] = retail_data["quantity"] * retail_data["price"]

In [20]:
# Extract year component
retail_data["invoice_year"] = retail_data["invoicedate"].dt.year

# Extract month component
retail_data["invoice_month"] = retail_data["invoicedate"].dt.month

# Extract year-month component
retail_data["year_month"] = retail_data["invoicedate"].dt.to_period("M")

# Extract quarter component
retail_data["invoice_quarter"] = retail_data["invoicedate"].dt.quarter

# Extract weekend indicator
retail_data["is_weekend"] = retail_data["invoicedate"].dt.dayofweek >= 5

# Extract day of month component
retail_data["invoice_day"] = retail_data["invoicedate"].dt.day

# Extract hour component
retail_data["invoice_hour"] = retail_data["invoicedate"].dt.hour

# Extract day of week component
retail_data["invoice_weekday"] = retail_data["invoicedate"].dt.weekday

In [21]:
# Create a new column to indicate if the quantity is negative
retail_data["is_negative_quantity"] = (retail_data["quantity"] < 0)

In [22]:
# Create a new column to indicate valid sales
retail_data["is_valid_sale"] = ((retail_data["is_cancellation"] == False) &
    (retail_data["quantity"] > 0) &
    (retail_data["price"] > 0))

In [23]:
retail_data['first_purchase_date'] = retail_data[retail_data['is_valid_sale'] == 1].dropna(subset=['customer_id']).groupby('customer_id')['invoicedate'].min()

In [12]:
order_summary = (
    retail_data[retail_data["is_valid_sale"] == True]
    .groupby("invoice")
    .agg(
        order_revenue=("revenue", "sum"),
        order_units=("quantity", "sum"),
        order_product_count=("stockcode", "nunique"),
        order_line_count=("stockcode", "size")
    )
    .reset_index()
)

In [26]:
retail_data[retail_data['first_purchase_date'].notna()]

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,is_cancellation,is_non_product,...,invoice_month,year_month,invoice_quarter,is_weekend,invoice_day,invoice_hour,invoice_weekday,is_negative_quantity,is_valid_sale,first_purchase_date
12346,490398,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,7,2009-12-06 10:37:00,2.55,17920,United Kingdom,False,False,...,12,2009-12,4,True,6,10,6,False,True,2009-12-14 08:34:00
12347,490398,16169C,WRAP BLUE REINDEER,25,2009-12-06 10:37:00,0.42,17920,United Kingdom,False,False,...,12,2009-12,4,True,6,10,6,False,True,2010-10-31 14:20:00
12348,490398,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,24,2009-12-06 10:37:00,3.39,17920,United Kingdom,False,False,...,12,2009-12,4,True,6,10,6,False,True,2010-09-27 14:59:00
12349,490398,37446,MINI CAKE STAND WITH HANGING CAKES,6,2009-12-06 10:37:00,1.45,17920,United Kingdom,False,False,...,12,2009-12,4,True,6,10,6,False,True,2010-04-29 13:20:00
12350,490398,20987,GEISHA GIRL CHOPSTICKS SET/5,5,2009-12-06 10:37:00,1.95,17920,United Kingdom,False,False,...,12,2009-12,4,True,6,10,6,False,True,2011-02-02 16:01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18283,490953,22171,3 HOOK PHOTO SHELF ANTIQUE WHITE,2,2009-12-08 14:56:00,8.50,13204,United Kingdom,False,False,...,12,2009-12,4,False,8,14,1,False,True,2010-02-19 17:16:00
18284,490953,22172,METAL SHELF WITH RAIL,4,2009-12-08 14:56:00,8.50,13204,United Kingdom,False,False,...,12,2009-12,4,False,8,14,1,False,True,2010-10-04 11:33:00
18285,490953,22173,METAL 4 HOOK HANGER FRENCH CHATEAU,8,2009-12-08 14:56:00,2.95,13204,United Kingdom,False,False,...,12,2009-12,4,False,8,14,1,False,True,2010-02-17 10:24:00
18286,490953,82483,WOOD 2 DRAWER CABINET WHITE FINISH,3,2009-12-08 14:56:00,5.95,13204,United Kingdom,False,False,...,12,2009-12,4,False,8,14,1,False,True,2009-12-16 10:45:00
